# Global aggregate 2050 storage rate: bootstrap histograms

Build the empirical distribution of the global aggregate 2050 storage
rate for every `(scenario, model)` combination, by bootstrap-convolving
the Part-1 Monte-Carlo samples across all participating countries.

Reference: Zhang et al. *Nat. Commun.* (2024), Fig. 3(b). There, for each
scenario she sums `target_2050` across 10 regions sample-by-sample
(paired index `i`), producing ~1 000 world totals per scenario.

Our Part-1 MC was run independently per country with feasibility
filtering, so `sample_idx` is not paired across countries (0% indices
appear in all 10 countries in `samples_reference.csv`). The
statistically correct treatment for independent regional MCs is a
bootstrap convolution: per (scenario, model), draw one `rate_2050` from
each country's empirical samples (with replacement), sum, and repeat
`N_BOOT` times to get the empirical distribution of the global aggregate
rate.

Within each scenario panel we overlay both growth models in the same
hue: solid = Logistic, dashed = Gompertz.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

os.environ.setdefault("MPLCONFIGDIR", '/private/tmp/mplconfig_iman2026')

# Locate repo root
for cand in [Path.cwd().resolve(), Path.cwd().resolve().parent,
             Path.cwd().resolve().parent.parent,
             Path('/Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026')]:
    if (cand / '01_growth_model').exists():
        ROOT = cand; break
else:
    raise FileNotFoundError('repo root with 01_growth_model/ not found')

P1   = ROOT / '01_growth_model' / 'output' / 'v8_2026-06-01'
OUT  = ROOT / '04_global_aggregate_2050' / 'output'
FIG  = OUT / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Config
SCENARIOS = ['minimum', 'reference', 'growth10', 'policy',
             'us1gt', 'maximum', 'ipcc_high', 'ipcc_low']
MODELS    = ['Logistic', 'Gompertz']
N_BOOT    = 10_000
SEED      = 42

# Same seaborn-deep palette used in 03 figure
SCENARIO_COLOR = {
    # Paul Tol "Muted" palette, colourblind-safe, print-safe (Tol 2018).
    'minimum':   '#888888',  # medium grey (neutral baseline, readable)
    'growth10':  '#DDCC77',  # sand
    'ipcc_low':  '#88CCEE',  # light blue
    'policy':    '#44AA99',  # teal
    'us1gt':     '#117733',  # green
    'reference': '#332288',  # indigo
    'ipcc_high': '#AA4499',  # purple
    'maximum':   '#CC6677',  # rose
}
SCENARIO_LABEL = {
    'minimum':'Minimum', 'reference':'Reference', 'growth10':'Growth 10%',
    'policy':'Policy', 'us1gt':'US 1 Gt', 'maximum':'Maximum',
    'ipcc_high':'IPCC High', 'ipcc_low':'IPCC Low',
}
MODEL_STYLE = {'Logistic': {'ls': '-',  'lw': 1.7, 'fill_alpha': 0.20},
               'Gompertz': {'ls': '--', 'lw': 1.7, 'fill_alpha': 0.00}}

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('repo :', ROOT)
print('out  :', OUT)
print('scenarios:', SCENARIOS)


In [ ]:
# 1. Bootstrap-convolve per (scenario, model)
rng = np.random.default_rng(SEED)
records, summary = [], []

for scen in SCENARIOS:
    p = P1 / f'samples_{scen}.csv'
    if not p.exists():
        print(f'  [skip] {scen}: no samples file')
        continue
    df = pd.read_csv(p)
    countries = sorted(df['Country'].unique())
    for mdl in MODELS:
        sub = df[df['Model'] == mdl]
        if sub.empty:
            continue
        # collect each country's rate_2050 array
        country_arrays = {c: sub.loc[sub['Country'] == c, 'rate_2050'].to_numpy()
                          for c in countries}
        country_arrays = {c: a for c, a in country_arrays.items() if len(a) > 0}
        if not country_arrays:
            continue
        # bootstrap convolution: draw one per country, sum, x N_BOOT
        world = np.zeros(N_BOOT)
        for c, arr in country_arrays.items():
            world += rng.choice(arr, size=N_BOOT, replace=True)
        records.extend(
            [{'scenario': scen, 'model': mdl, 'rate_2050_gt_yr': float(w)}
             for w in world])
        summary.append({
            'scenario': scen, 'model': mdl,
            'n_countries': len(country_arrays),
            'countries': ','.join(country_arrays.keys()),
            'median': float(np.median(world)),
            'mean':   float(world.mean()),
            'std':    float(world.std()),
            'p5':     float(np.percentile(world, 5)),
            'p25':    float(np.percentile(world, 25)),
            'p75':    float(np.percentile(world, 75)),
            'p95':    float(np.percentile(world, 95)),
        })

records_df = pd.DataFrame(records)
summary_df = pd.DataFrame(summary)
records_df.to_csv(OUT / 'global_aggregate_2050_bootstrap.csv', index=False)
summary_df.to_csv(OUT / 'global_aggregate_2050_summary.csv', index=False)

print(f'bootstrap rows : {len(records_df):,}')
print('\n=== Summary (Gt/yr) ===')
print(summary_df[['scenario','model','n_countries','median','mean','p5','p95']]
      .round(2).to_string(index=False))

xmax = float(np.percentile(records_df['rate_2050_gt_yr'].to_numpy(), 99.5))
print(f'xmax (99.5%ile) = {xmax:.2f} Gt/yr')


In [ ]:
# 2. Pre-screening 2050 rate distribution (single panel)
# Per-scenario 32-bin histograms with the top dominance ribbon.
# Same bootstrap data as the post-screening figure; the peak-year strip
# has moved into the mirror plot (alongside the pre/post peak-year
# comparison).
from scipy.stats import gaussian_kde
from scipy.signal import medfilt

fig, ax = plt.subplots(figsize=(13.5, 7.0))

# Dominance ribbon: KDE per scenario, median-filtered to drop artefacts
fine_x = np.linspace(0.01, xmax, 300)
dens   = np.zeros((len(SCENARIOS), len(fine_x)))
for i, scen in enumerate(SCENARIOS):
    v = records_df[records_df['scenario']==scen]['rate_2050_gt_yr'].to_numpy()
    if v.size > 1:
        dens[i] = gaussian_kde(v, bw_method=0.10)(fine_x)
dominant_idx = medfilt(np.argmax(dens, axis=0), kernel_size=11).astype(int)
runs = []
cur_i, cur_x0 = dominant_idx[0], fine_x[0]
for j in range(1, len(fine_x)):
    if dominant_idx[j] != cur_i:
        runs.append((cur_i, cur_x0, fine_x[j-1])); cur_i, cur_x0 = dominant_idx[j], fine_x[j]
runs.append((cur_i, cur_x0, fine_x[-1]))

# Per-scenario 32-bin histograms
max_count_global = 0
for scen in SCENARIOS:
    col = SCENARIO_COLOR[scen]
    sub_scen = records_df[records_df['scenario'] == scen]
    if sub_scen.empty: continue
    s_lo = float(sub_scen['rate_2050_gt_yr'].min())
    s_hi = float(sub_scen['rate_2050_gt_yr'].max())
    pad  = max((s_hi - s_lo) * 0.04, 1e-3)
    lo, hi = max(0.0, s_lo - pad), s_hi + pad
    bins   = np.linspace(lo, hi, 33)
    bin_ctr = 0.5 * (bins[:-1] + bins[1:])
    for mdl in MODELS:
        v = sub_scen[sub_scen['model'] == mdl]['rate_2050_gt_yr'].to_numpy()
        if v.size == 0: continue
        counts, _ = np.histogram(v, bins=bins)
        if counts.max() == 0: continue
        max_count_global = max(max_count_global, int(counts.max()))
        sty = MODEL_STYLE[mdl]
        if sty['fill_alpha'] > 0:
            ax.fill_between(bin_ctr, counts, step='mid', color=col,
                              alpha=sty['fill_alpha'], linewidth=0)
        ax.stairs(counts, bins, color=col, linestyle=sty['ls'],
                   linewidth=sty['lw'], alpha=0.92, baseline=None)

# Top dominance ribbon
ribbon_y0 = max_count_global * 1.06
ribbon_y1 = max_count_global * 1.13
for j in range(len(fine_x) - 1):
    sc = SCENARIOS[dominant_idx[j]]
    ax.fill_betweenx([ribbon_y0, ribbon_y1], fine_x[j], fine_x[j+1],
                      color=SCENARIO_COLOR[sc], alpha=0.85, linewidth=0)
y_levels = [ribbon_y1 + max_count_global * 0.015,
            ribbon_y1 + max_count_global * 0.075]
for k, (sc_idx, x0, x1) in enumerate(runs):
    if x1 - x0 < 0.10: continue
    sc = SCENARIOS[sc_idx]; xc = 0.5 * (x0 + x1)
    y_lab = y_levels[k % 2]
    ax.text(xc, y_lab, SCENARIO_LABEL[sc],
             ha='center', va='bottom', fontsize=8.0, fontweight='bold',
             color=SCENARIO_COLOR[sc])

ax.set_xlim(0, xmax)
ax.set_ylim(0, max_count_global * 1.20)
ax.set_xlabel('Aggregated total global storage rate for 2050 (Gt/yr)')
ax.set_ylabel('Frequency')
ax.grid(axis='y', color='#eeeeee', lw=0.5)
ax.set_axisbelow(True)

model_handles = [
    Line2D([0],[0], color='#444', ls='-',  lw=1.8, label='Logistic'),
    Line2D([0],[0], color='#444', ls='--', lw=1.8, label='Gompertz'),
]
leg_b1 = ax.legend(handles=model_handles,
                    bbox_to_anchor=(1.025, 1.0), loc='upper left',
                    frameon=False, title='Growth model',
                    title_fontsize=9, fontsize=9)
ax.add_artist(leg_b1)
scen_handles = []
for scen in SCENARIOS:
    s = summary_df[summary_df['scenario'] == scen]
    if s.empty: continue
    mL = float(s.loc[s['model']=='Logistic','median'].values[0]) if (s['model']=='Logistic').any() else np.nan
    mG = float(s.loc[s['model']=='Gompertz','median'].values[0]) if (s['model']=='Gompertz').any() else np.nan
    nc = int(s['n_countries'].max())
    scen_handles.append(
        Line2D([0],[0], color=SCENARIO_COLOR[scen], lw=2.4,
                label=f'{SCENARIO_LABEL[scen]:<10s}  L={mL:.1f}  G={mG:.1f}  [n={nc}]'))
ax.legend(handles=scen_handles,
           bbox_to_anchor=(1.025, 0.78), loc='upper left',
           frameon=False,
           title='Scenario  (median Gt/yr, n countries)',
           title_fontsize=9, fontsize=8.6)

fig.suptitle('Pre-screening global aggregate 2050 storage rate distribution',
              fontsize=14, fontweight='bold', y=0.985)

plt.subplots_adjust(top=0.91, bottom=0.10, left=0.07, right=0.78)

png = FIG / 'fig_global_aggregate_2050_histograms.png'
pdf = FIG / 'fig_global_aggregate_2050_histograms.pdf'
fig.savefig(png, dpi=200, bbox_inches='tight')
fig.savefig(pdf, dpi=240, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {png}')
print(f'Saved: {pdf}')
